# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  
**Name:** Simi Chakravarty
---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [47]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [48]:
df['revenue'] = df['qty'] * df['price']

total_revenue = df['revenue'].sum()
total_units = df['qty'].sum()

print(f'total revenue: ${total_revenue:.2f}')
print(f'total units: {total_units}')
print(f'number of rows: {len(df)}')

total revenue: $8520.00
total units: 783
number of rows: 400


### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [49]:
by_category = df.groupby('category', as_index = False)['revenue'].sum().sort_values(by = 'revenue', ascending = False)
by_category['revenue_percentage'] = by_category['revenue'] / total_revenue * 100
by_category

,category,revenue,revenue_percentage
1,Food,4293.0,50.387324
2,Merch,1771.5,20.792254
0,Drink,1554.0,18.239437
3,RainGear,901.5,10.580986


### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [50]:
vendor_average = df.groupby('vendor_id', as_index = False)['revenue'].agg(['mean', 'count']).sort_values(by = 'mean', ascending = False).rename(columns = {'mean': 'average_revenue', 'count': 'order_count'})
vendor_average

,vendor_id,average_revenue,order_count
0,V-01,22.595745,94
3,V-18,21.750000,108
1,V-05,20.580645,93
2,V-10,20.314286,105


### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [51]:
merch_revenue = by_category[by_category['category'] == 'Merch']['revenue_percentage'].round(1)
print(f'Merch Revenue Share: {merch_revenue.iloc[0]}%')

Merch Revenue Share: 20.8%


### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [52]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

joined = df.merge(vendor_names, how = 'left', on = 'vendor_id', validate = 'many_to_one')

# Proof that the row count and total revenue didn't change
print(f'old number of rows: {len(df)}')
print(f'old total revenue: ${df["revenue"].sum():.2f}')
print(f'new number of rows: {len(joined)}')
print(f'new total revenue: ${joined["revenue"].sum():.2f}')

# V-18 does not have a vendor name listed
# I will label the name as 'Unknown Vendor'

joined['vendor_name'] = joined['vendor_name'].fillna('Unknown Vendor')
joined

old number of rows: 400
old total revenue: $8520.00
new number of rows: 400
new total revenue: $8520.00


,vendor_id,category,qty,price,revenue,vendor_name
0,V-10,Drink,2,24.0,48.0,Cav Merch North
1,V-18,RainGear,1,12.0,12.0,Unknown Vendor
2,V-18,Drink,3,4.5,13.5,Unknown Vendor
3,V-10,Food,2,12.0,24.0,Cav Merch North
4,V-18,Drink,3,7.5,22.5,Unknown Vendor
...,...,...,...,...,...,...
395,V-18,Merch,1,12.0,12.0,Unknown Vendor
396,V-01,Merch,2,24.0,48.0,Hoos Burgers
397,V-10,Food,3,7.5,22.5,Cav Merch North
398,V-18,Merch,2,24.0,48.0,Unknown Vendor


**The unmatched vendor, and what I did about it:** _..._

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [53]:
pivot_table = pd.pivot_table(joined, values = 'revenue', index = 'vendor_name', columns = 'category', aggfunc = 'sum', margins = True, margins_name = 'Total')
pivot_table

category,Drink,Food,Merch,RainGear,Total
vendor_name,,,,,
Cav Merch North,502.5,1054.5,400.5,175.5,2133.0
Hoos Burgers,171.0,1338.0,373.5,241.5,2124.0
Rotunda Tacos,298.5,882.0,489.0,244.5,1914.0
Unknown Vendor,582.0,1018.5,508.5,240.0,2349.0
Total,1554.0,4293.0,1771.5,901.5,8520.0


### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [54]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

_your answer here_